# WordPress画像整備の優先記事分析

公開中のWordPress記事を対象に、GA4のページビューを主指標として上位20件を選び、Search Consoleの検索流入と現在の画像実装を突合する。

- 主期間: 2025-07-20〜2026-07-19（直近12か月、GA4とGSCの共通確定期間）
- 補助期間: 2026-04-21〜2026-07-19（直近90日）
- 選定基準: 公開記事の12か月GA4ページビュー降順
- GSC: ページ単位で全行取得し、1,000行制限のある既存クエリ集計は順位付けに使用しない
- 画像監査: 共通アイキャッチと固定要素画像を除外して記事固有画像を数える


In [1]:
from __future__ import annotations

import csv
import html
import json
import os
import re
from collections import Counter
from pathlib import Path
from urllib.parse import quote, urlparse

import pandas as pd
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from google.auth.transport.requests import Request
from google.oauth2 import service_account
from googleapiclient.discovery import build
from requests.auth import HTTPBasicAuth

ROOT = next(candidate for candidate in [Path.cwd(), *Path.cwd().parents] if (candidate / '.env').exists() and (candidate / 'analyze_content_seo.py').exists())
OUT_DIR = ROOT / '03_研究資料・レビュー/2026-07-22_画像整備優先記事分析'
RAW_DIR = OUT_DIR / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)

END_DATE = '2026-07-19'
START_12M = '2025-07-20'
START_90D = '2026-04-21'
DAYS_12M = 365
DAYS_90D = 90

load_dotenv(ROOT / '.env')
required = ['WP_URL', 'WP_USER', 'WP_APP_PASSWORD', 'GA4_PROPERTY_ID', 'GOOGLE_APPLICATION_CREDENTIALS', 'GSC_SITE_URL']
missing = [key for key in required if not os.getenv(key)]
assert not missing, f'設定不足: {missing}'
assert Path(os.environ['GOOGLE_APPLICATION_CREDENTIALS']).is_file(), '認証JSONが見つからない'
print('設定確認: OK（秘密値は表示しない）')

設定確認: OK（秘密値は表示しない）


## データ取得

GA4はローカルのprotobuf互換性に影響されないREST APIを使う。Search Consoleはページ集計を25,000行ずつページングする。WordPressは認証付きREST APIを読み取り専用で取得する。

In [2]:
def post_id_from_path(value: str | None) -> str | None:
    if not value:
        return None
    path = urlparse(value).path if '://' in value else value.split('?', 1)[0]
    match = re.search(r'/(\d+)/?$', path)
    return match.group(1) if match else None

def fetch_ga4_pages(start_date: str, end_date: str) -> tuple[pd.DataFrame, dict]:
    creds = service_account.Credentials.from_service_account_file(
        os.environ['GOOGLE_APPLICATION_CREDENTIALS'],
        scopes=['https://www.googleapis.com/auth/analytics.readonly'],
    )
    creds.refresh(Request())
    endpoint = (
        'https://analyticsdata.googleapis.com/v1beta/properties/'
        f"{os.environ['GA4_PROPERTY_ID']}:runReport"
    )
    metrics = ['screenPageViews', 'activeUsers', 'sessions', 'engagedSessions']
    body = {
        'dateRanges': [{'startDate': start_date, 'endDate': end_date}],
        'dimensions': [{'name': 'pagePath'}],
        'metrics': [{'name': name} for name in metrics],
        'limit': '100000',
        'offset': '0',
        'orderBys': [{'metric': {'metricName': 'screenPageViews'}, 'desc': True}],
    }
    response = requests.post(
        endpoint,
        headers={'Authorization': f'Bearer {creds.token}', 'Content-Type': 'application/json'},
        json=body, timeout=60,
    )
    response.raise_for_status()
    payload = response.json()
    rows = []
    for row in payload.get('rows', []):
        metric_values = row.get('metricValues', [])
        item = {'pagePath': row['dimensionValues'][0]['value']}
        for index, name in enumerate(metrics):
            item[name] = int(float(metric_values[index]['value']))
        item['post_id'] = post_id_from_path(item['pagePath'])
        rows.append(item)
    frame = pd.DataFrame(rows)
    meta = {
        'start_date': start_date, 'end_date': end_date,
        'api_row_count': int(payload.get('rowCount', len(rows))),
        'returned_rows': len(rows), 'metric_names': metrics,
    }
    assert meta['api_row_count'] == meta['returned_rows'], 'GA4行が取り切れていない'
    return frame, meta

def fetch_gsc(start_date: str, end_date: str, dimensions: list[str], max_rows: int = 200000) -> tuple[pd.DataFrame, dict]:
    creds = service_account.Credentials.from_service_account_file(
        os.environ['GOOGLE_APPLICATION_CREDENTIALS'],
        scopes=['https://www.googleapis.com/auth/webmasters.readonly'],
    )
    service = build('searchconsole', 'v1', credentials=creds, cache_discovery=False)
    rows, start_row, page_size = [], 0, 25000
    while start_row < max_rows:
        request = {
            'startDate': start_date, 'endDate': end_date,
            'dimensions': dimensions, 'rowLimit': page_size, 'startRow': start_row,
        }
        payload = service.searchanalytics().query(
            siteUrl=os.environ['GSC_SITE_URL'], body=request
        ).execute()
        batch = payload.get('rows', [])
        for row in batch:
            item = dict(zip(dimensions, row.get('keys', [])))
            item.update({key: row.get(key, 0) for key in ['clicks', 'impressions', 'ctr', 'position']})
            if 'page' in item:
                item['post_id'] = post_id_from_path(item['page'])
            rows.append(item)
        if len(batch) < page_size:
            break
        start_row += page_size
    meta = {
        'start_date': start_date, 'end_date': end_date, 'dimensions': dimensions,
        'returned_rows': len(rows), 'hit_safety_cap': len(rows) >= max_rows,
    }
    assert not meta['hit_safety_cap'], 'GSC安全上限に到達。上限を見直す必要がある'
    return pd.DataFrame(rows), meta

def fetch_wp_posts() -> list[dict]:
    endpoint = os.environ['WP_URL'].rstrip('/') + '/wp-json/wp/v2/posts'
    auth = HTTPBasicAuth(os.environ['WP_USER'], os.environ['WP_APP_PASSWORD'])
    fields = 'id,title,slug,link,date,modified,featured_media,content,excerpt'
    posts, page, expected_total = [], 1, None
    while True:
        response = requests.get(
            endpoint, auth=auth, timeout=60,
            params={'status': 'publish', 'context': 'edit', 'per_page': 100, 'page': page, '_fields': fields},
        )
        response.raise_for_status()
        if expected_total is None:
            expected_total = int(response.headers.get('X-WP-Total', 0))
        batch = response.json()
        posts.extend(batch)
        if len(posts) >= expected_total or not batch:
            break
        page += 1
    assert len(posts) == expected_total
    return posts

def fetch_media(media_ids: set[int]) -> dict[int, dict]:
    endpoint = os.environ['WP_URL'].rstrip('/') + '/wp-json/wp/v2/media'
    auth = HTTPBasicAuth(os.environ['WP_USER'], os.environ['WP_APP_PASSWORD'])
    result = {}
    ids = sorted(i for i in media_ids if i)
    for start in range(0, len(ids), 100):
        chunk = ids[start:start + 100]
        response = requests.get(
            endpoint, auth=auth, timeout=60,
            params={'include': ','.join(map(str, chunk)), 'per_page': 100, '_fields': 'id,source_url,alt_text,media_details,slug'},
        )
        response.raise_for_status()
        for item in response.json():
            result[int(item['id'])] = item
    return result

In [3]:
ga4_12m, ga4_meta_12m = fetch_ga4_pages(START_12M, END_DATE)
ga4_90d, ga4_meta_90d = fetch_ga4_pages(START_90D, END_DATE)
gsc_pages_12m, gsc_meta_12m = fetch_gsc(START_12M, END_DATE, ['page'])
gsc_pages_90d, gsc_meta_90d = fetch_gsc(START_90D, END_DATE, ['page'])
gsc_query_pages_12m, gsc_queries_meta = fetch_gsc(START_12M, END_DATE, ['query', 'page'])
wp_posts = fetch_wp_posts()

ga4_12m.to_csv(RAW_DIR / 'ga4_pages_12m.csv', index=False)
ga4_90d.to_csv(RAW_DIR / 'ga4_pages_90d.csv', index=False)
gsc_pages_12m.to_csv(RAW_DIR / 'gsc_pages_12m.csv', index=False)
gsc_pages_90d.to_csv(RAW_DIR / 'gsc_pages_90d.csv', index=False)
gsc_query_pages_12m.to_csv(RAW_DIR / 'gsc_query_pages_12m.csv', index=False)

source_meta = {
    'ga4_12m': ga4_meta_12m, 'ga4_90d': ga4_meta_90d,
    'gsc_pages_12m': gsc_meta_12m, 'gsc_pages_90d': gsc_meta_90d,
    'gsc_query_pages_12m': gsc_queries_meta, 'wp_public_posts': len(wp_posts),
}
(OUT_DIR / 'source_metadata.json').write_text(json.dumps(source_meta, ensure_ascii=False, indent=2), encoding='utf-8')
source_meta

{'ga4_12m': {'start_date': '2025-07-20',
  'end_date': '2026-07-19',
  'api_row_count': 411,
  'returned_rows': 411,
  'metric_names': ['screenPageViews',
   'activeUsers',
   'sessions',
   'engagedSessions']},
 'ga4_90d': {'start_date': '2026-04-21',
  'end_date': '2026-07-19',
  'api_row_count': 224,
  'returned_rows': 224,
  'metric_names': ['screenPageViews',
   'activeUsers',
   'sessions',
   'engagedSessions']},
 'gsc_pages_12m': {'start_date': '2025-07-20',
  'end_date': '2026-07-19',
  'dimensions': ['page'],
  'returned_rows': 140,
  'hit_safety_cap': False},
 'gsc_pages_90d': {'start_date': '2026-04-21',
  'end_date': '2026-07-19',
  'dimensions': ['page'],
  'returned_rows': 89,
  'hit_safety_cap': False},
 'gsc_query_pages_12m': {'start_date': '2025-07-20',
  'end_date': '2026-07-19',
  'dimensions': ['query', 'page'],
  'returned_rows': 1496,
  'hit_safety_cap': False},
 'wp_public_posts': 218}

## WordPress画像監査

メディアID 314は217記事で再利用される共通アイキャッチ。本文ではID 361（著者）、629（固定フッター）、487（旧共通施術写真）を固定画像として除外する。さらに全記事で20回以上使われる画像URLはテンプレート候補として除外し、上位20件は後で個別確認する。

In [4]:
def clean_text(value: str) -> str:
    return BeautifulSoup(html.unescape(value or ''), 'html.parser').get_text(' ', strip=True)

def parse_img_media_id(tag) -> int | None:
    classes = tag.get('class', [])
    if isinstance(classes, str):
        classes = classes.split()
    for cls in classes:
        match = re.fullmatch(r'wp-image-(\d+)', cls)
        if match:
            return int(match.group(1))
    return None

parsed_posts = []
all_image_ids, all_image_srcs = [], []
for post in wp_posts:
    rendered = post.get('content', {}).get('rendered', '')
    raw = post.get('content', {}).get('raw', rendered)
    soup = BeautifulSoup(rendered, 'html.parser')
    images = []
    for tag in soup.find_all('img'):
        src = (tag.get('src') or tag.get('data-src') or '').split('?', 1)[0]
        media_id = parse_img_media_id(tag)
        if not src and media_id is None:
            continue
        images.append({'media_id': media_id, 'src': src, 'alt': clean_text(tag.get('alt', ''))})
        if media_id:
            all_image_ids.append(media_id)
        if src:
            all_image_srcs.append(src)
    parsed_posts.append({
        'post_id': str(post['id']), 'title': clean_text(post.get('title', {}).get('rendered', '')),
        'slug': post.get('slug', ''), 'link': post.get('link', ''),
        'date': post.get('date', ''), 'modified': post.get('modified', ''),
        'featured_media': int(post.get('featured_media') or 0),
        'images': images, 'h2_count': len(soup.find_all('h2')), 'raw_content': raw,
    })

featured_counts = Counter(p['featured_media'] for p in parsed_posts)
body_id_counts = Counter(all_image_ids)
body_src_counts = Counter(all_image_srcs)
fixed_media_ids = {361, 487, 629}
frequent_srcs = {src for src, count in body_src_counts.items() if count >= 20}
media_ids = {p['featured_media'] for p in parsed_posts if p['featured_media']} | set(all_image_ids)
media_map = fetch_media(media_ids)

audit_rows = []
for post in parsed_posts:
    article_images = []
    fixed_images = []
    seen = set()
    for image in post['images']:
        key = (image['media_id'], image['src'])
        if key in seen:
            continue
        seen.add(key)
        is_fixed = image['media_id'] in fixed_media_ids or image['src'] in frequent_srcs
        (fixed_images if is_fixed else article_images).append(image)
    feature_id = post['featured_media']
    feature = media_map.get(feature_id, {})
    feature_reuse = featured_counts.get(feature_id, 0) if feature_id else 0
    feature_needs_replacement = feature_id == 0 or feature_reuse >= 20
    body_gap = max(0, 3 - len(article_images))
    audit_rows.append({
        'post_id': post['post_id'], 'title': post['title'], 'slug': post['slug'], 'link': post['link'],
        'date': post['date'], 'modified': post['modified'], 'featured_media': feature_id,
        'featured_url': feature.get('source_url', ''), 'featured_reuse_count': feature_reuse,
        'featured_needs_replacement': feature_needs_replacement,
        'total_body_images': len(post['images']), 'fixed_body_images': len(fixed_images),
        'article_specific_body_images': len(article_images),
        'article_image_ids': ','.join(str(x['media_id'] or '') for x in article_images),
        'article_image_urls': ' | '.join(x['src'] for x in article_images),
        'body_images_to_add_for_target_3': body_gap, 'h2_count': post['h2_count'],
    })

wp_audit = pd.DataFrame(audit_rows)
wp_audit.to_csv(RAW_DIR / 'wp_published_image_audit.csv', index=False)
print('公開記事:', len(wp_audit))
print('アイキャッチID分布:', featured_counts.most_common(5))
print('本文の頻出メディアID:', body_id_counts.most_common(8))
print('記事固有画像ゼロ:', int((wp_audit.article_specific_body_images == 0).sum()))

公開記事: 218
アイキャッチID分布: [(314, 217), (0, 1)]
本文の頻出メディアID: [(361, 218), (629, 184), (487, 68), (844, 7), (673, 3), (549, 3), (437, 3), (1032, 2)]
記事固有画像ゼロ: 134


## 結合と順位付け

同一記事の複数パス表記は投稿IDで集約する。12か月PVが同数の場合は、90日PV、12か月GSCクリックの順で並べる。直近勢いは、90日の日平均PVを12か月の日平均PVで割った参考値であり、主順位には使わない。

In [5]:
def aggregate_ga4(frame: pd.DataFrame, suffix: str) -> pd.DataFrame:
    numeric = frame[frame['post_id'].notna()].copy()
    grouped = numeric.groupby('post_id', as_index=False)[['screenPageViews', 'activeUsers', 'sessions', 'engagedSessions']].sum()
    return grouped.rename(columns={c: f'{c}_{suffix}' for c in grouped.columns if c != 'post_id'})

def aggregate_gsc(frame: pd.DataFrame, suffix: str) -> pd.DataFrame:
    numeric = frame[frame['post_id'].notna()].copy()
    numeric['weighted_position'] = numeric['position'] * numeric['impressions']
    grouped = numeric.groupby('post_id', as_index=False).agg(
        clicks=('clicks', 'sum'), impressions=('impressions', 'sum'),
        weighted_position=('weighted_position', 'sum'),
    )
    grouped['ctr'] = grouped['clicks'] / grouped['impressions'].where(grouped['impressions'] != 0)
    grouped['position'] = grouped['weighted_position'] / grouped['impressions'].where(grouped['impressions'] != 0)
    grouped = grouped.drop(columns=['weighted_position'])
    return grouped.rename(columns={c: f'gsc_{c}_{suffix}' for c in grouped.columns if c != 'post_id'})

ranked = wp_audit.copy()
for frame in [aggregate_ga4(ga4_12m, '12m'), aggregate_ga4(ga4_90d, '90d'),
              aggregate_gsc(gsc_pages_12m, '12m'), aggregate_gsc(gsc_pages_90d, '90d')]:
    ranked = ranked.merge(frame, on='post_id', how='left', validate='one_to_one')

metric_cols = [c for c in ranked.columns if any(token in c for token in ['screenPageViews_', 'activeUsers_', 'sessions_', 'engagedSessions_', 'gsc_clicks_', 'gsc_impressions_'])]
ranked[metric_cols] = ranked[metric_cols].fillna(0)
ranked['recent_pace_index'] = (ranked['screenPageViews_90d'] / DAYS_90D) / (ranked['screenPageViews_12m'] / DAYS_12M).replace(0, pd.NA)
ranked['gsc_ctr_12m'] = ranked['gsc_ctr_12m'].fillna(0)
ranked['gsc_ctr_90d'] = ranked['gsc_ctr_90d'].fillna(0)
ranked = ranked.sort_values(['screenPageViews_12m', 'screenPageViews_90d', 'gsc_clicks_12m'], ascending=False).reset_index(drop=True)
ranked['access_rank'] = ranked.index + 1

# 上位記事の主要検索語（クリック、次に表示回数で上位5件）
query_rows = gsc_query_pages_12m[gsc_query_pages_12m['post_id'].notna()].copy()
query_rows = query_rows.sort_values(['post_id', 'clicks', 'impressions'], ascending=[True, False, False])
top_query_map = {}
for post_id, group in query_rows.groupby('post_id'):
    values = []
    for _, row in group.head(5).iterrows():
        values.append(f"{row['query']}（{int(row['clicks'])}クリック）")
    top_query_map[str(post_id)] = ' / '.join(values)
ranked['top_search_queries_12m'] = ranked['post_id'].map(top_query_map).fillna('')

def image_action(row) -> str:
    actions = []
    if row['featured_needs_replacement']:
        actions.append('記事別アイキャッチへ差し替え')
    gap = int(row['body_images_to_add_for_target_3'])
    if gap:
        actions.append(f'本文画像を{gap}枚追加')
    elif row['article_specific_body_images'] >= 3:
        actions.append('既存本文画像を目視確認')
    return ' + '.join(actions)

ranked['recommended_image_action'] = ranked.apply(image_action, axis=1)
ranked.to_csv(OUT_DIR / '公開記事全件ランキング.csv', index=False)
top20 = ranked.head(20).copy()
top20.to_csv(OUT_DIR / '画像整備優先20記事.csv', index=False)

# 将来の差し込み作業用に、上位20記事の取得時点の生HTMLを保存
top20_ids = set(top20['post_id'])
snapshot = []
for post in parsed_posts:
    if post['post_id'] in top20_ids:
        snapshot.append({key: post[key] for key in ['post_id', 'title', 'slug', 'link', 'date', 'modified', 'h2_count', 'raw_content']})
(RAW_DIR / 'top20_wordpress_content_snapshot.json').write_text(
    json.dumps(snapshot, ensure_ascii=False, indent=2), encoding='utf-8'
)

preview_cols = ['access_rank', 'post_id', 'title', 'screenPageViews_12m', 'screenPageViews_90d', 'gsc_clicks_12m', 'recent_pace_index', 'article_specific_body_images', 'recommended_image_action']
top20[preview_cols]

,access_rank,post_id,title,screenPageViews_12m,screenPageViews_90d,gsc_clicks_12m,recent_pace_index,article_specific_body_images,recommended_image_action
0,1,976,【脊柱側弯症手術後の痛み – 半数以上が経験する長期的な苦痛への理解を深める】,1570.0,495.0,876.0,1.278662,0,記事別アイキャッチへ差し替え + 本文画像を3枚追加
1,2,965,側弯症の装具4種類を解説！ミルウォーキー・ボストン・シュロスの違いと選び方,889.0,210.0,109.0,0.958005,4,記事別アイキャッチへ差し替え + 既存本文画像を目視確認
2,3,441,【パルス電磁場療法（PEMF）の概要と解説】,838.0,123.0,274.0,0.595267,0,記事別アイキャッチへ差し替え + 本文画像を3枚追加
3,4,1176,【側弯症解説シリーズ】第5回：意外と知らない！脊柱側弯症と股関節・骨盤の密接な関係！,733.0,319.0,297.0,1.764969,1,記事別アイキャッチへ差し替え + 本文画像を2枚追加
4,5,1182,【側弯症解説シリーズ】第7回：脊柱側弯症で変わる歩き方と背中の筋肉の働き〜気づかない身体の変...,543.0,288.0,201.0,2.151013,0,記事別アイキャッチへ差し替え + 本文画像を3枚追加
5,6,1092,大人の脊柱側弯症、手術が必要なときって？―分かりやすく解説します,534.0,57.0,0.0,0.432896,2,記事別アイキャッチへ差し替え + 本文画像を1枚追加
6,7,1171,【側弯症解説シリーズ】第3回：脊柱を支える筋肉の秘密！側弯症で起こる筋肉の拘縮メカニズムとセ...,335.0,81.0,151.0,0.980597,0,記事別アイキャッチへ差し替え + 本文画像を3枚追加
7,8,923,【腰痛：骨盤ベルトって本当に効果あるの？ 筋肉への影響とバランス能力向上について解説！】,267.0,39.0,0.0,0.592385,0,記事別アイキャッチへ差し替え + 本文画像を3枚追加
8,9,1145,背中や腰の痛み…大人の脊柱側弯症ってどうすればいいの？楽になるケアと整体施術のヒミツ,192.0,11.0,0.0,0.23235,1,記事別アイキャッチへ差し替え + 本文画像を2枚追加
9,10,469,【脊柱側弯症の中学生のための治療法と注意点】,191.0,72.0,0.0,1.528796,1,記事別アイキャッチへ差し替え + 本文画像を2枚追加


## データ品質検証

順位を意思決定に使う前に、期間の包含関係、重複、公開状態、API行の取り切り、URL結合率、画像監査の基本整合性を確認する。

In [6]:
assert len(wp_audit) == 218
assert wp_audit['post_id'].is_unique and wp_audit['link'].is_unique
assert len(top20) == 20 and top20['post_id'].is_unique
assert top20['screenPageViews_12m'].is_monotonic_decreasing
assert (ranked['screenPageViews_90d'] <= ranked['screenPageViews_12m']).all()
assert (ranked['gsc_clicks_90d'] <= ranked['gsc_clicks_12m']).all()
assert (ranked[metric_cols] >= 0).all().all()
assert set(top20['post_id']).issubset(set(wp_audit['post_id']))
assert ga4_meta_12m['api_row_count'] == ga4_meta_12m['returned_rows']
assert not gsc_meta_12m['hit_safety_cap'] and not gsc_queries_meta['hit_safety_cap']

wp_ids = set(wp_audit['post_id'])
ga4_ids = set(ga4_12m.loc[ga4_12m['post_id'].notna(), 'post_id'])
gsc_ids = set(gsc_pages_12m.loc[gsc_pages_12m['post_id'].notna(), 'post_id'])
matched_ga4 = wp_ids & ga4_ids
matched_gsc = wp_ids & gsc_ids
numeric_ga4_views = ga4_12m.loc[ga4_12m['post_id'].notna(), 'screenPageViews'].sum()
matched_ga4_views = ga4_12m.loc[ga4_12m['post_id'].isin(wp_ids), 'screenPageViews'].sum()
quality_summary = {
    'wp_public_posts': len(wp_ids),
    'ga4_rows_12m': len(ga4_12m),
    'gsc_page_rows_12m': len(gsc_pages_12m),
    'gsc_query_page_rows_12m': len(gsc_query_pages_12m),
    'wp_posts_matched_to_ga4': len(matched_ga4),
    'wp_post_count_ga4_coverage': len(matched_ga4) / len(wp_ids),
    'wp_posts_matched_to_gsc': len(matched_gsc),
    'wp_post_count_gsc_coverage': len(matched_gsc) / len(wp_ids),
    'matched_share_of_numeric_ga4_views': matched_ga4_views / numeric_ga4_views if numeric_ga4_views else None,
    'generic_featured_posts': int((wp_audit['featured_reuse_count'] >= 20).sum()),
    'no_featured_posts': int((wp_audit['featured_media'] == 0).sum()),
    'no_article_specific_body_images': int((wp_audit['article_specific_body_images'] == 0).sum()),
    'top20_views_12m': int(top20['screenPageViews_12m'].sum()),
    'all_public_post_views_12m': int(ranked['screenPageViews_12m'].sum()),
    'top20_share_of_public_post_views': float(top20['screenPageViews_12m'].sum() / ranked['screenPageViews_12m'].sum()),
}
(OUT_DIR / 'data_quality_summary.json').write_text(json.dumps(quality_summary, ensure_ascii=False, indent=2), encoding='utf-8')
quality_summary

{'wp_public_posts': 218,
 'ga4_rows_12m': 411,
 'gsc_page_rows_12m': 140,
 'gsc_query_page_rows_12m': 1496,
 'wp_posts_matched_to_ga4': 194,
 'wp_post_count_ga4_coverage': 0.8899082568807339,
 'wp_posts_matched_to_gsc': 93,
 'wp_post_count_gsc_coverage': 0.42660550458715596,
 'matched_share_of_numeric_ga4_views': 0.9686330061722149,
 'generic_featured_posts': 217,
 'no_featured_posts': 1,
 'no_article_specific_body_images': 134,
 'top20_views_12m': 7460,
 'all_public_post_views_12m': 9573,
 'top20_share_of_public_post_views': 0.7792750443956963}

## 結論の読み方

- 上位20件は「現在公開されている投稿」の12か月PV順位。固定ページ・トップページ・非公開化されたURLは除外している。
- GSCは検索流入の補助情報であり、PVとの合成スコアは作っていない。
- 直近90日の勢いは参考値。テレビ放映など単発要因の可能性があるため、順位を自動的に入れ替えていない。
- 画像の枚数は、共通アイキャッチID 314、著者画像361、旧共通施術画像487、フッター画像629を記事固有画像から除外している。
- 頻出画像の機械判定は上位20件で個別確認してからWordPressへ反映する。
